# Hands-On-Activity 7.2 | CNN Architectures from Scratch and Transfer Learning

## Procedure

1. Create a VGG-16 Network from scratch.

2. Train on the given [dataset](https://www.kaggle.com/datasets/samrat230599/fastai-imagenet/data). and reach at least 95% Accuracy. Do not use pre-trained weights.

3. Plot the accuracy for both Training and Validation. Include a confusion matrix, and a classification report (Precision, Recall , F1 score, Sensitivity) Plot the ROC AUC score result of the model

4. Interpret and evaluate the result of the model.

5. Import a pretrained VGG-16 Network, adjust the final layers to match the dataset. Must have pre-trained weights.

6. Fine tune on the given dataset, plot the same metrics in #3 and #4.

7. Compare the results of the trained network from scratch and fine-tuned network.

### Import Data

In [26]:
from google.colab import drive
drive.mount('/content/drive')
data_dir = '/content/drive/MyDrive/SCHOOLFILES/CPE32/CPE313/hoa7.2/FastAI_ImageNet_v2'

# 3. Check it looks right
import os
print(os.listdir(data_dir))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['Training.csv', 'Validation.csv', 'training_composition.txt', 'validation_composition.txt', 'train', 'val']


In [27]:
# Fix — rename extensions to lowercase
import pathlib

count = 0
for split in ['train', 'val']:
    for cls in os.listdir(f'{data_dir}/{split}'):
        cls_path = f'{data_dir}/{split}/{cls}'
        for f in pathlib.Path(cls_path).iterdir():
            if f.suffix == '.JPEG':
                f.rename(f.with_suffix('.jpeg'))
                count += 1

print(f"Renamed {count} files")

Renamed 0 files


In [28]:
import torch
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((320, 320)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Dataset already has a train/val split — use it directly
train_set = ImageFolder(root=f'{data_dir}/train', transform=transform)
val_set   = ImageFolder(root=f'{data_dir}/val',   transform=transform)

train_loader = DataLoader(train_set, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_set,   batch_size=32, shuffle=False, num_workers=2)

# Verify
print(f"Classes:            {train_set.classes}")
print(f"Training samples:   {len(train_set)}")
print(f"Validation samples: {len(val_set)}")

images, labels = next(iter(train_loader))
print(f"Batch shape: {images.shape}")   # → torch.Size([32, 3, 320, 320])

Classes:            ['cassette_player', 'chain_saw', 'church', 'english_springer', 'french_horn', 'garbage_truck', 'gas_pump', 'golf_ball', 'parachute', 'tench']
Training samples:   9469
Validation samples: 3899
Batch shape: torch.Size([32, 3, 320, 320])


In [29]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


### VGG-16 from scratch

In [30]:
import torch.nn as nn

class VGG16Scratch(nn.Module):
    def __init__(self, num_classes=10):
        super(VGG16Scratch, self).__init__()

        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            # Block 2
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            # Block 3
            nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            # Block 4
            nn.Conv2d(256, 512, kernel_size=3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            # Block 5
            nn.Conv2d(512, 512, kernel_size=3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1), nn.BatchNorm2d(512), nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        self.avgpool = nn.AdaptiveAvgPool2d((7, 7))

        self.classifier = nn.Sequential(
            nn.Linear(512 * 7 * 7, 4096), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(4096, 4096),        nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(4096, num_classes),
        )

        self._initialize_weights()

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None: nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

model_scratch = VGG16Scratch(num_classes=10).to(device)
print(f"Total parameters: {sum(p.numel() for p in model_scratch.parameters()):,}")

Total parameters: 134,309,962


In [36]:
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_scratch.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, epochs=50):
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    total_train = len(train_loader.dataset)
    total_val   = len(val_loader.dataset)

    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1} -------------------------------")

        # ── Training ──
        model.train()
        train_loss, train_correct, total = 0, 0, 0

        for batch, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss    += loss.item() * images.size(0)
            train_correct += outputs.argmax(1).eq(labels).sum().item()
            total         += images.size(0)

            # Print every ~3 batches (adjust the interval as you like)
            if (batch + 1) % 3 == 0 or total == total_train:
                print(f"  loss: {loss.item():.6f}  [{total:>4}/{total_train}]")

        train_loss /= total_train
        train_acc   = train_correct / total_train

        # ── Validation ──
        model.eval()
        val_loss, val_correct = 0, 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss    = criterion(outputs, labels)

                val_loss    += loss.item() * images.size(0)
                val_correct += outputs.argmax(1).eq(labels).sum().item()

        val_loss /= total_val
        val_acc   = val_correct / total_val

        scheduler.step(val_acc)

        # ── Epoch Summary ──
        print(f"Train:\n  Accuracy: {train_acc*100:.1f}%  |  Avg loss: {train_loss:.6f}")
        print(f"Val:\n  Accuracy: {val_acc*100:.1f}%  |  Avg loss: {val_loss:.6f}")

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        if val_acc >= 0.95:
            print(f"\nReached 95% validation accuracy at epoch {epoch+1}!")
            torch.save(model.state_dict(), 'mobilenetv2_scratch_best.pth')
            break

    return history

history_scratch = train_model(
    model_scratch, train_loader, val_loader,
    criterion, optimizer, scheduler, epochs=10
)


Epoch 1 -------------------------------
  loss: 3.337362  [  96/9469]
  loss: 2.927259  [ 192/9469]
  loss: 2.635674  [ 288/9469]
  loss: 2.246778  [ 384/9469]
  loss: 2.737949  [ 480/9469]
  loss: 3.186135  [ 576/9469]
  loss: 3.823541  [ 672/9469]
  loss: 2.217703  [ 768/9469]
  loss: 2.242828  [ 864/9469]
  loss: 2.649358  [ 960/9469]
  loss: 2.246410  [1056/9469]
  loss: 2.349942  [1152/9469]
  loss: 2.409356  [1248/9469]
  loss: 2.176033  [1344/9469]
  loss: 2.263558  [1440/9469]
  loss: 2.412808  [1536/9469]
  loss: 2.252738  [1632/9469]
  loss: 2.310252  [1728/9469]
  loss: 2.551465  [1824/9469]
  loss: 2.202216  [1920/9469]
  loss: 2.313027  [2016/9469]
  loss: 2.283404  [2112/9469]
  loss: 2.358507  [2208/9469]


KeyboardInterrupt: 

#### Evaluation

In [ ]:
import matplotlib.pyplot as plt

def plot_history(history, title='VGG-16 from Scratch'):
    epochs = range(1, len(history['train_acc']) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(epochs, [a*100 for a in history['train_acc']], label='Train Accuracy', color='steelblue')
    ax1.plot(epochs, [a*100 for a in history['val_acc']],   label='Val Accuracy',   color='darkorange')
    ax1.set_title(f'{title} — Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy (%)')
    ax1.legend()
    ax1.grid(True)

    ax2.plot(epochs, history['train_loss'], label='Train Loss', color='steelblue')
    ax2.plot(epochs, history['val_loss'],   label='Val Loss',   color='darkorange')
    ax2.set_title(f'{title} — Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.show()

plot_history(history_scratch, title='VGG-16 from Scratch')

In [ ]:
import numpy as np
import seaborn as sns
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_auc_score, roc_curve)
from sklearn.preprocessing import label_binarize
import torch.nn.functional as F

def evaluate_model(model, val_loader, class_names, title='Model'):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            outputs = model(images)
            probs   = F.softmax(outputs, dim=1).cpu().numpy()
            preds   = outputs.argmax(1).cpu().numpy()

            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    all_labels = np.array(all_labels)
    all_preds  = np.array(all_preds)
    all_probs  = np.array(all_probs)

    # ── Confusion Matrix ──
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'{title} — Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

    # ── Classification Report ──
    print(f"\n{title} — Classification Report\n")
    print(classification_report(all_labels, all_preds, target_names=class_names))

    # ── ROC-AUC ──
    n_classes = len(class_names)
    labels_bin = label_binarize(all_labels, classes=range(n_classes))

    plt.figure(figsize=(10, 7))
    for i, cls in enumerate(class_names):
        fpr, tpr, _ = roc_curve(labels_bin[:, i], all_probs[:, i])
        auc = roc_auc_score(labels_bin[:, i], all_probs[:, i])
        plt.plot(fpr, tpr, label=f'{cls} (AUC = {auc:.3f})')

    macro_auc = roc_auc_score(labels_bin, all_probs, average='macro', multi_class='ovr')
    plt.plot([0, 1], [0, 1], 'k--', linewidth=1)
    plt.title(f'{title} — ROC-AUC Curve (Macro AUC = {macro_auc:.3f})')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.legend(loc='lower right', fontsize=8)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    return all_labels, all_preds, all_probs

class_names = train_set.classes
labels_s, preds_s, probs_s = evaluate_model(
    model_scratch, val_loader, class_names, title='VGG-16 from Scratch'
)

### Pre-trained VGG-16 network

#### Import data

In [41]:
import torch
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Dataset already has a train/val split — use it directly
train_set = ImageFolder(root=f'{data_dir}/train', transform=transform)
val_set   = ImageFolder(root=f'{data_dir}/val',   transform=transform)

train_loader = DataLoader(train_set, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_set,   batch_size=32, shuffle=False, num_workers=2)

# Verify
print(f"Classes:            {train_set.classes}")
print(f"Training samples:   {len(train_set)}")
print(f"Validation samples: {len(val_set)}")

images, labels = next(iter(train_loader))
print(f"Batch shape: {images.shape}")   # → torch.Size([32, 3, 320, 320])

Classes:            ['cassette_player', 'chain_saw', 'church', 'english_springer', 'french_horn', 'garbage_truck', 'gas_pump', 'golf_ball', 'parachute', 'tench']
Training samples:   9469
Validation samples: 3925
Batch shape: torch.Size([32, 3, 224, 224])


#### Data Preprocessing

Since VGG-16 is designed for 224px input images, and the dataset contains 320px images, the image data needs to be preprocessed.

####Pre-trained model

In [42]:
import torchvision.models as models

# Load pretrained VGG-16
model_pretrained = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)

# Freeze all feature extractor layers
for param in model_pretrained.features.parameters():
    param.requires_grad = False

# Replace classifier to match 10 classes
model_pretrained.classifier = nn.Sequential(
    nn.Linear(512 * 7 * 7, 4096), nn.ReLU(inplace=True), nn.Dropout(0.5),
    nn.Linear(4096, 4096),        nn.ReLU(inplace=True), nn.Dropout(0.5),
    nn.Linear(4096, 10),
)

model_pretrained = model_pretrained.to(device)

# Only train the classifier
optimizer_pt = optim.Adam(model_pretrained.classifier.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler_pt = ReduceLROnPlateau(optimizer_pt, mode='max', patience=3, factor=0.5)

print("Pretrained VGG-16 loaded. Classifier layers unfrozen for fine-tuning.")
print(f"Trainable parameters: {sum(p.numel() for p in model_pretrained.parameters() if p.requires_grad):,}")

Pretrained VGG-16 loaded. Classifier layers unfrozen for fine-tuning.
Trainable parameters: 119,586,826


In [ ]:
history_pretrained = train_model(
    model_pretrained, train_loader, val_loader,
    criterion, optimizer_pt, scheduler_pt, epochs=20
)

plot_history(history_pretrained, title='VGG-16 Pretrained (Fine-Tuned)')


Epoch 1 -------------------------------
  loss: 1.866293  [  96/9469]
  loss: 1.669048  [ 192/9469]
  loss: 0.840207  [ 288/9469]
  loss: 0.587406  [ 384/9469]
  loss: 0.377923  [ 480/9469]
  loss: 0.416451  [ 576/9469]
  loss: 0.298467  [ 672/9469]
  loss: 0.189721  [ 768/9469]
  loss: 0.400169  [ 864/9469]
  loss: 0.348625  [ 960/9469]
  loss: 0.426690  [1056/9469]
  loss: 0.275433  [1152/9469]
  loss: 0.161824  [1248/9469]
  loss: 0.118945  [1344/9469]
  loss: 0.789477  [1440/9469]
  loss: 0.172118  [1536/9469]
  loss: 0.258301  [1632/9469]
  loss: 0.524368  [1728/9469]
  loss: 0.067990  [1824/9469]
  loss: 0.198096  [1920/9469]
  loss: 0.153934  [2016/9469]
  loss: 0.096922  [2112/9469]
  loss: 0.154263  [2208/9469]
  loss: 0.198344  [2304/9469]
  loss: 0.310936  [2400/9469]
  loss: 0.243269  [2496/9469]
  loss: 0.304677  [2592/9469]
  loss: 0.104895  [2688/9469]
  loss: 0.012780  [2784/9469]
  loss: 0.322026  [2880/9469]
  loss: 0.113391  [2976/9469]
  loss: 0.130593  [3072/9469]

#### Evaluation

In [ ]:
labels_pt, preds_pt, probs_pt = evaluate_model(
    model_pretrained, val_loader, class_names, title='VGG-16 Pretrained (Fine-Tuned)'
)

### Comparing Both Models

In [ ]:
from sklearn.metrics import accuracy_score

acc_scratch    = accuracy_score(labels_s,  preds_s)
acc_pretrained = accuracy_score(labels_pt, preds_pt)

labels_bin = label_binarize(labels_s, classes=range(10))
auc_scratch    = roc_auc_score(labels_bin, probs_s,  average='macro', multi_class='ovr')
auc_pretrained = roc_auc_score(labels_bin, probs_pt, average='macro', multi_class='ovr')

# ── Summary Table ──
print("=" * 50)
print(f"{'Metric':<25} {'Scratch':>10} {'Pretrained':>12}")
print("=" * 50)
print(f"{'Val Accuracy':<25} {acc_scratch*100:>9.2f}% {acc_pretrained*100:>11.2f}%")
print(f"{'Macro ROC-AUC':<25} {auc_scratch:>10.4f} {auc_pretrained:>12.4f}")
print(f"{'Epochs Trained':<25} {len(history_scratch['train_acc']):>10} {len(history_pretrained['train_acc']):>12}")
print("=" * 50)

# ── Side-by-side Accuracy Curves ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, history, label in zip(axes,
                               [history_scratch, history_pretrained],
                               ['Scratch', 'Pretrained']):
    epochs = range(1, len(history['train_acc']) + 1)
    ax.plot(epochs, [a*100 for a in history['train_acc']], label='Train', color='steelblue')
    ax.plot(epochs, [a*100 for a in history['val_acc']],   label='Val',   color='darkorange')
    ax.axhline(y=95, color='red', linestyle='--', linewidth=1, label='95% target')
    ax.set_title(f'VGG-16 {label}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy (%)')
    ax.legend()
    ax.grid(True)

plt.suptitle('Scratch vs. Pretrained — Accuracy Comparison', fontsize=13)
plt.tight_layout()
plt.show()

## Supplementary

1. Create a MobileNetV2 Network.

2. Train on the given dataset and reach at least 95% Accuracy. Do not use pre-trained weights.

3. Plot the accuracy for both Training and Validation. Include a confusion matrix, and a classification report (Precision, Recall , F1 score, Sensitivity) Plot the ROC AUC score result of the model

4. Interpret and evaluate the result of the model.

5. Import a pretrained MobileNetV2 Network, adjust the final layers to match the dataset. Must have pre-trained weights.

6. Fine tune on the given dataset, plot the same metrics in #3 and #4.

7. Compare the results of the trained network from scratch and fine-tuned network.

In [ ]:
import os
import torch
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(224, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_set = ImageFolder(root=f'{data_dir}/train', transform=transform_train)
val_set   = ImageFolder(root=f'{data_dir}/val',   transform=transform_val)

train_loader = DataLoader(train_set, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"Classes:            {train_set.classes}")
print(f"Training samples:   {len(train_set)}")
print(f"Validation samples: {len(val_set)}")

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

### MobileNetV2 from Scratch

In [ ]:
import torch.nn as nn

# ── Depthwise Separable Conv Block ──────────────────────────────────────
class ConvBNReLU(nn.Sequential):
    def __init__(self, in_ch, out_ch, kernel=3, stride=1, groups=1):
        padding = (kernel - 1) // 2
        super().__init__(
            nn.Conv2d(in_ch, out_ch, kernel, stride=stride,
                      padding=padding, groups=groups, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU6(inplace=True)
        )

# ── Inverted Residual Block (the core building block of MobileNetV2) ────
class InvertedResidual(nn.Module):
    def __init__(self, in_ch, out_ch, stride, expand_ratio):
        super().__init__()
        self.use_residual = (stride == 1 and in_ch == out_ch)
        hidden = int(round(in_ch * expand_ratio))

        layers = []
        if expand_ratio != 1:
            layers.append(ConvBNReLU(in_ch, hidden, kernel=1))  # pointwise expand
        layers += [
            ConvBNReLU(hidden, hidden, stride=stride, groups=hidden),  # depthwise
            nn.Conv2d(hidden, out_ch, 1, bias=False),                  # pointwise project
            nn.BatchNorm2d(out_ch),
        ]
        self.conv = nn.Sequential(*layers)

    def forward(self, x):
        return x + self.conv(x) if self.use_residual else self.conv(x)

# ── Full MobileNetV2 ─────────────────────────────────────────────────────
class MobileNetV2Scratch(nn.Module):
    # Each row: [expand_ratio, out_channels, num_blocks, stride]
    CFG = [
        [1,  16, 1, 1],
        [6,  24, 2, 2],
        [6,  32, 3, 2],
        [6,  64, 4, 2],
        [6,  96, 3, 1],
        [6, 160, 3, 2],
        [6, 320, 1, 1],
    ]

    def __init__(self, num_classes=10):
        super().__init__()

        # First conv layer
        self.features = [ConvBNReLU(3, 32, stride=2)]

        # Inverted residual blocks
        in_ch = 32
        for t, c, n, s in self.CFG:
            for i in range(n):
                stride = s if i == 0 else 1
                self.features.append(InvertedResidual(in_ch, c, stride, t))
                in_ch = c

        # Last conv layer
        self.features.append(ConvBNReLU(320, 1280, kernel=1))
        self.features = nn.Sequential(*self.features)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(1280, num_classes),
        )

        self._initialize_weights()

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None: nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

model_scratch = MobileNetV2Scratch(num_classes=10).to(device)
print(f"Total parameters: {sum(p.numel() for p in model_scratch.parameters()):,}")
# → ~3.4 million (vs VGG-16's ~138 million)

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_scratch.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5, verbose=True)

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler,
                epochs=50, save_path='best_model.pth', target_acc=0.95):
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0

    for epoch in range(epochs):
        # ── Training ──
        model.train()
        train_loss, train_correct, total = 0, 0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss    += loss.item() * images.size(0)
            train_correct += outputs.argmax(1).eq(labels).sum().item()
            total         += images.size(0)

        train_loss /= total
        train_acc   = train_correct / total

        # ── Validation ──
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss    = criterion(outputs, labels)

                val_loss    += loss.item() * images.size(0)
                val_correct += outputs.argmax(1).eq(labels).sum().item()
                val_total   += images.size(0)

        val_loss /= val_total
        val_acc   = val_correct / val_total

        scheduler.step(val_acc)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(f"Epoch [{epoch+1:02d}/{epochs}] "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}% | "
              f"Val Loss: {val_loss:.4f}   | Val Acc: {val_acc*100:.2f}%")

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)

        if val_acc >= target_acc:
            print(f"\n✅ Reached {target_acc*100:.0f}% validation accuracy at epoch {epoch+1}!")
            break

    print(f"\nBest Val Accuracy: {best_val_acc*100:.2f}%")
    return history

history_scratch = train_model(
    model_scratch, train_loader, val_loader,
    criterion, optimizer, scheduler,
    epochs=10, save_path='mobilenetv2_scratch_best.pth'
)

#### Evaluation

In [ ]:
import matplotlib.pyplot as plt

def plot_history(history, title='MobileNetV2'):
    epochs = range(1, len(history['train_acc']) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(epochs, [a*100 for a in history['train_acc']], label='Train Accuracy', color='steelblue')
    ax1.plot(epochs, [a*100 for a in history['val_acc']],   label='Val Accuracy',   color='darkorange')
    ax1.axhline(y=95, color='red', linestyle='--', linewidth=1, label='95% target')
    ax1.set_title(f'{title} — Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy (%)')
    ax1.legend()
    ax1.grid(True)

    ax2.plot(epochs, history['train_loss'], label='Train Loss', color='steelblue')
    ax2.plot(epochs, history['val_loss'],   label='Val Loss',   color='darkorange')
    ax2.set_title(f'{title} — Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.show()

plot_history(history_scratch, title='MobileNetV2 from Scratch')

In [ ]:
import numpy as np
import seaborn as sns
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_auc_score, roc_curve)
from sklearn.preprocessing import label_binarize
import torch.nn.functional as F

def evaluate_model(model, val_loader, class_names, title='Model'):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            outputs = model(images)
            probs   = F.softmax(outputs, dim=1).cpu().numpy()
            preds   = outputs.argmax(1).cpu().numpy()

            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    all_labels = np.array(all_labels)
    all_preds  = np.array(all_preds)
    all_probs  = np.array(all_probs)

    # ── Confusion Matrix ──────────────────────────────────────────────────
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'{title} — Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

    # ── Classification Report ─────────────────────────────────────────────
    print(f"\n{title} — Classification Report\n")
    print(classification_report(all_labels, all_preds, target_names=class_names))

    # ── ROC-AUC Curve ─────────────────────────────────────────────────────
    n_classes  = len(class_names)
    labels_bin = label_binarize(all_labels, classes=range(n_classes))

    plt.figure(figsize=(10, 7))
    for i, cls in enumerate(class_names):
        fpr, tpr, _ = roc_curve(labels_bin[:, i], all_probs[:, i])
        auc = roc_auc_score(labels_bin[:, i], all_probs[:, i])
        plt.plot(fpr, tpr, label=f'{cls} (AUC = {auc:.3f})')

    macro_auc = roc_auc_score(labels_bin, all_probs, average='macro', multi_class='ovr')
    plt.plot([0, 1], [0, 1], 'k--', linewidth=1)
    plt.title(f'{title} — ROC-AUC Curve (Macro AUC = {macro_auc:.3f})')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.legend(loc='lower right', fontsize=8)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    return all_labels, all_preds, all_probs

class_names = train_set.classes
labels_s, preds_s, probs_s = evaluate_model(
    model_scratch, val_loader, class_names, title='MobileNetV2 from Scratch'
)

### Pretrained MobileNetV2

In [ ]:
import torchvision.models as models

# Load pretrained MobileNetV2
model_pretrained = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)

# Freeze all feature extractor layers
for param in model_pretrained.features.parameters():
    param.requires_grad = False

# Replace the classifier head to match 10 classes
# MobileNetV2's classifier is model.classifier[1] (Linear 1280 → 1000)
model_pretrained.classifier = nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(model_pretrained.last_channel, 10),
)

model_pretrained = model_pretrained.to(device)

# Only train the new classifier head
optimizer_pt = optim.Adam(model_pretrained.classifier.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler_pt = ReduceLROnPlateau(optimizer_pt, mode='max', patience=3, factor=0.5, verbose=True)

trainable = sum(p.numel() for p in model_pretrained.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model_pretrained.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,}")

#### Finetuning

In [ ]:
history_pretrained = train_model(
    model_pretrained, train_loader, val_loader,
    criterion, optimizer_pt, scheduler_pt,
    epochs=20, save_path='mobilenetv2_pretrained_best.pth'
)

plot_history(history_pretrained, title='MobileNetV2 Pretrained (Fine-Tuned)')

#### Evaluation

In [ ]:
labels_pt, preds_pt, probs_pt = evaluate_model(
    model_pretrained, val_loader, class_names, title='MobileNetV2 Pretrained (Fine-Tuned)'
)

### Compare and Contrast the two models

In [ ]:
from sklearn.metrics import accuracy_score

acc_scratch    = accuracy_score(labels_s,  preds_s)
acc_pretrained = accuracy_score(labels_pt, preds_pt)

labels_bin     = label_binarize(labels_s, classes=range(10))
auc_scratch    = roc_auc_score(labels_bin, probs_s,  average='macro', multi_class='ovr')
auc_pretrained = roc_auc_score(labels_bin, probs_pt, average='macro', multi_class='ovr')

# ── Summary Table ──────────────────────────────────────────────────────
print("=" * 52)
print(f"{'Metric':<25} {'Scratch':>10} {'Pretrained':>14}")
print("=" * 52)
print(f"{'Val Accuracy':<25} {acc_scratch*100:>9.2f}% {acc_pretrained*100:>13.2f}%")
print(f"{'Macro ROC-AUC':<25} {auc_scratch:>10.4f} {auc_pretrained:>14.4f}")
print(f"{'Epochs Trained':<25} {len(history_scratch['train_acc']):>10} {len(history_pretrained['train_acc']):>14}")
print(f"{'Total Params':<25} {'~3.4M':>10} {'~3.4M':>14}")
print("=" * 52)

# ── Side-by-side Accuracy Curves ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, history, label in zip(axes,
                               [history_scratch, history_pretrained],
                               ['Scratch', 'Pretrained (Fine-Tuned)']):
    epochs = range(1, len(history['train_acc']) + 1)
    ax.plot(epochs, [a*100 for a in history['train_acc']], label='Train', color='steelblue')
    ax.plot(epochs, [a*100 for a in history['val_acc']],   label='Val',   color='darkorange')
    ax.axhline(y=95, color='red', linestyle='--', linewidth=1, label='95% target')
    ax.set_title(f'MobileNetV2 {label}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy (%)')
    ax.legend()
    ax.grid(True)

plt.suptitle('MobileNetV2: Scratch vs. Pretrained — Accuracy Comparison', fontsize=13)
plt.tight_layout()
plt.show()

# Conclusion